# Download Data from kaggle

In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"mhmirghaderi","key":"1eec1364ddc0754ccc36335b11440ea5"}'}

In [2]:

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:

!kaggle competitions download -c data-science-5-sbu

  0% 0.00/48.2M [00:00<?, ?B/s]
100% 48.2M/48.2M [00:00<00:00, 1.17GB/s]


In [4]:
!unzip -q data-science-5-sbu.zip


#Preparing and Loading Data

In [5]:
# Import necessary libraries
!pip install optuna
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_log_error
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error
# --- Load Data ---
# Make sure the Kaggle dataset files are in the same directory as this notebook
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('x_test.csv')
    sample_submission_df = pd.read_csv('sample_submission.csv')
except FileNotFoundError:
    print("Error: Make sure 'train.csv', 'x_test.csv', and 'sample_submission.csv' are in the correct directory.")
    # Exit or create dummy data if needed for the script to run
    # For demonstration, creating dummy dataframes if files are not found
    train_df = pd.DataFrame()
    test_df = pd.DataFrame()

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 17.1 MB/s eta 0:00:00
Training data shape: (1100000, 21)
Test data shape: (100000, 20)


#Exploratory Data Analysis (EDA)

In [23]:
if not train_df.empty:
    print("\n--- Data Exploration (First 5 Rows) ---")
    print(train_df.head())

    # --- Target Variable Analysis ---
    # The RMSLE metric suggests the target is skewed. We'll use log1p transformation.
    # log1p(x) is equivalent to log(x + 1) and is useful for values that could be 0.
    train_df['Policy Cost'] = np.log1p(train_df['Policy Cost'])

    # --- Feature Separation (Numerical vs Categorical) ---
    categorical_features = train_df.select_dtypes(include=['object', 'category']).columns
    numerical_features = train_df.select_dtypes(include=np.number).columns.drop(['ID', 'Policy Cost'])

    print(f"\nCategorical Features: {list(categorical_features)}")
    print(f"Numerical Features: {list(numerical_features)}")

    # --- Missing Value Check ---
    print("\n--- Missing Values per Column ---")
    missing_values = train_df.isnull().sum()
    print(missing_values[missing_values > 0])


--- Data Exploration (First 5 Rows) ---
   ID  Years Lived  Sex  Yearly Earnings Relationship Status  Dependent Count  \
0   0         26.0  Man           8071.0           Unmarried              NaN   
1   1         42.0  Man             49.0           Unmarried              2.0   
2   2         52.0  Man          80793.0              Spouse              2.0   
3   3         33.0  Man          30663.0           Unmarried              1.0   
4   4         34.0  Man          59638.0              Spouse              0.0   

  Academic Standing   Job Title  Wellness Index       Region  ...  \
0         Secondary         NaN       17.927378         City  ...   
1     Undergraduate     Jobless       49.828507         City  ...   
2         Doctorate     Jobless       27.539608      Exurban  ...   
3     Undergraduate  Freelancer       24.355045  Countryside  ...   
4     Undergraduate         NaN       33.394840  Countryside  ...   

  Prior Claims  Automobile Age  Financial Rating  Coverag

#Feature Engineering

In [32]:
def feature_engineer(df):
    """Applies feature engineering steps to the dataframe."""

    # --- Temporal Features ---
    # Assuming 'Date of Birth' is the date column mentioned in the assignment.
    # It seems to be the most relevant date column available.
    if 'Date of Birth' in df.columns:
        df['Date of Birth'] = pd.to_datetime(df['Date of Birth'], errors='coerce')
        df['Birth_Year'] = df['Date of Birth'].dt.year
        df['Birth_Month'] = df['Date of Birth'].dt.month
        df['Birth_DayOfWeek'] = df['Date of Birth'].dt.dayofweek
        df.drop('Date of Birth', axis=1, inplace=True)


    df['Earnings_per_Dependent'] = np.where(
            df['Dependent Count'] > 0,
            df['Yearly Earnings'] / df['Dependent Count'],
            df['Yearly Earnings']
    )
    df['Income_to_Financial_Rating_Ratio'] = df['Yearly Earnings'] / (df['Financial Rating'] + 1)
    df['Wellness_per_Year_Lived'] = df['Wellness Index'] / (df['Years Lived'] + 1)
    df['Claim_Frequency'] = df['Prior Claims'] / (df['Coverage Period'] + 1)
    df['Family_Obligation_Index'] = df['Dependent Count'] / (df['Yearly Earnings'] + 1)
    df['Composite_Risk_Score'] = (df['Automobile Age'] * df['Prior Claims']) / (df['Financial Rating'] + 1)
    df['Wellness_x_YearsLived'] = df['Wellness Index'] * df['Years Lived']
    df['Earnings_Bin'] = pd.qcut(df['Yearly Earnings'], q=5, labels=False, duplicates='drop')

    df['Financial_Wellness_Interaction'] = df['Financial Rating'] * df['Wellness Index']
    df['Age_Adjusted_Income_Quality'] = df['Years Lived'] * df['Income_to_Financial_Rating_Ratio']
    df['Low_Income_High_Claims_Risk'] = df['Claim_Frequency'] / (df['Yearly Earnings'] + 1)
    df['Financial_Rating_Squared'] = df['Financial Rating'] ** 2
    for col in df.select_dtypes(include=np.number).columns:
        df[col] = df[col].fillna(df[col].median())

    for col in df.select_dtypes(include=['object', 'category']).columns:
        df[col] = df[col].fillna(df[col].mode()[0])

    # --- Categorical Feature Encoding ---
    # Using Label Encoding as it is simple and effective for tree-based models
    for col in df.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

    return df

# Apply feature engineering to both train and test sets
if not train_df.empty:
    train_processed = feature_engineer(train_df.copy())
    test_processed = feature_engineer(test_df.copy())

    print("\n--- Processed Data (First 5 Rows) ---")
    print(train_processed.head())


--- Processed Data (First 5 Rows) ---
   ID  Years Lived  Sex  Yearly Earnings  Relationship Status  \
0   0         26.0    0           8071.0                    2   
1   1         42.0    0             49.0                    2   
2   2         52.0    0          80793.0                    1   
3   3         33.0    0          30663.0                    2   
4   4         34.0    0          59638.0                    1   

   Dependent Count  Academic Standing  Job Title  Wellness Index  Region  ...  \
0              2.0                  2          2       17.927378       0  ...   
1              2.0                  3          1       49.828507       0  ...   
2              2.0                  0          1       27.539608       2  ...   
3              1.0                  3          0       24.355045       1  ...   
4              0.0                  3          2       33.394840       1  ...   

   Wellness_per_Year_Lived  Claim_Frequency  Family_Obligation_Index  \
0          

# Traninig the model and Prediction

In [33]:
# یک مدل اولیه آموزش بده
features1 = [col for col in train_processed.columns if col not in ['ID', 'Policy Cost']]
target1 = 'Policy Cost'
X = train_processed[features1]
y = train_processed[target1]
X_test = test_processed[features1]

initial_model = lgb.LGBMRegressor(random_state=42)
initial_model.fit(X, y)

# اهمیت ویژگی‌ها را استخراج کن
feature_importances = pd.DataFrame({
    'feature': X.columns,
    'importance': initial_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importances.head(50))

# فقط 50 ویژگی برتر را انتخاب کن
top_features = feature_importances['feature'].head(4).tolist()

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.105273 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3726
[LightGBM] [Info] Number of data points in the train set: 1100000, number of used features: 31
[LightGBM] [Info] Start training from score 2.014977
                             feature  importance
2                    Yearly Earnings         388
12                  Financial Rating         381
7                     Wellness Index         325
14             Coverage Commencement         216
25             Wellness_x_YearsLived         169
19            Earnings_per_Dependent         143
27    Financial_Wellness_Interaction         143
10                      Prior Claims         132
21           Wellness_per_Year_Lived         124
20  Income_to_Financia

In [34]:

features1 = [col for col in train_processed.columns if col not in ['ID', 'Policy Cost']]
target1 = 'Policy Cost'

X = train_processed[top_features]
y = train_processed[target1]
X_test = test_processed[top_features]

# 4. Train final model with best params
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_predictions = np.zeros(X.shape[0])
test_predictions = np.zeros(X_test.shape[0])
best_params={'learning_rate': 0.014887871141600159,
             'num_leaves': 64,
             'max_depth': 5,
             'lambda_l1': 4.4266547526661494e-05,
             'lambda_l2': 0.0008588681460274362,
             'feature_fraction': 0.6102806682152676,
             'bagging_fraction': 0.7343780054540967,
             'bagging_freq': 6}
for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
    print(f"===== Final Training Fold {fold+1} =====")
    X_train, y_train = X.iloc[train_index], y.iloc[train_index]
    X_val, y_val = X.iloc[val_index], y.iloc[val_index]

    # The pruning_callback is REMOVED from here
    model = lgb.LGBMRegressor(**best_params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              eval_metric='rmse',
              # Only keep the early stopping callback
              callbacks=[lgb.early_stopping(100, verbose=False)])

    oof_predictions[val_index] = model.predict(X_val)
    # I also corrected this line to divide by the correct number of splits (3)
    test_predictions += model.predict(X_test) / kf.get_n_splits()
train_processed['oof_preds'] = oof_predictions
train_processed['error'] = train_processed['Policy Cost'] - train_processed['oof_preds']
# Final score and submission file
final_rmsle = np.sqrt(mean_squared_log_error(np.expm1(y), np.expm1(oof_predictions)))
print(f"\nOverall OOF RMSLE with Tuned Model: {final_rmsle}")

===== Final Training Fold 1 =====
[LightGBM] [Warning] feature_fraction is set=0.6102806682152676, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6102806682152676
[LightGBM] [Warning] lambda_l2 is set=0.0008588681460274362, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0008588681460274362
[LightGBM] [Warning] lambda_l1 is set=4.4266547526661494e-05, reg_alpha=0.0 will be ignored. Current value: lambda_l1=4.4266547526661494e-05
[LightGBM] [Warning] bagging_fraction is set=0.7343780054540967, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7343780054540967
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] feature_fraction is set=0.6102806682152676, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6102806682152676
[LightGBM] [Warning] lambda_l2 is set=0.00

In [35]:
if not train_df.empty:
    # --- Create Submission File ---
    # Convert predictions back from log scale
    final_predictions = np.expm1(test_predictions)

    # Ensure predictions are non-negative
    final_predictions[final_predictions < 0] = 0

    submission_df = pd.DataFrame({'ID': test_df['ID'], 'Policy Cost': final_predictions})
    submission_df.to_csv('Submission6.csv', index=False)

    print("\n'submission.csv' file created successfully!")
    print(submission_df.head())


'submission.csv' file created successfully!
   ID  Policy Cost
0   0     6.586300
1   1     6.689224
2   2     6.498920
3   3     6.553077
4   4     6.672714
